# Scalable Construction and Machine Learning Analysis of a VSX–TESS Variable Star Dataset

This notebook is a project snapshot summarizing the current state of the VSX–TESS variable star classification project.

It combines:

- data pipeline construction
- VSX–TIC matching
- SPOC / QLP / TESSCut recovery
- trend detection and conditional detrending
- feature extraction
- Logistic Regression and Random Forest analysis
- provenance-dependent findings

The notebook is written as a research narrative with executable analysis placeholders. Paths should be updated to match the local project directory before running.

## 0. Project Configuration

Update these paths before running the notebook locally.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Update these paths locally
PROJECT_ROOT = Path("/data/projects/TESS-research")

# Metadata / pipeline files
METADATA_PARQUET = PROJECT_ROOT / "data_pipeline" / "TESSAugmented_QC.parquet"
TREND_PARQUET = PROJECT_ROOT / "trend_detection" / "TESSAugmented_QC_trend.parquet"
DETREND_PARQUET = PROJECT_ROOT / "detrending" / "TESSAugmented_QC_tesscut_conditional_detrended.parquet"

# Feature / ML outputs
FEATURE_PARQUET = PROJECT_ROOT / "feature_extraction" / "TESS_features.parquet"
ANALYSIS_OUTPUT_DIR = PROJECT_ROOT / "ml_analytics" / "output"

# Figure output folder
FIGURE_DIR = PROJECT_ROOT / "paper_writing" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Figure dir:", FIGURE_DIR)

Project root: /data/projects/TESS-research
Figure dir: /data/projects/TESS-research/paper_figures


# 1. Abstract

We present an end-to-end research pipeline for large-scale variable star classification using photometric observations from the NASA TESS mission and labels from the AAVSO VSX catalog. A major challenge in this problem is the incomplete availability of high-quality TESS light curves and the ambiguity in mapping VSX catalog objects to TESS TIC identifiers.

To address these issues, we developed a scalable multi-stage pipeline that combines multi-candidate TIC crossmatching, hierarchical light curve recovery using SPOC, QLP, and TESSCut products, and automated feature extraction using time-series analysis techniques including Lomb–Scargle periodograms.

The pipeline improves usable light curve coverage from approximately 12% under naive nearest-neighbor mapping to approximately 96.5% through the integration of TESSCut-based recovery. Trend detection and conditional detrending were investigated as a possible way to improve TESSCut performance. However, downstream Random Forest experiments showed little improvement after conditional detrending, suggesting that TESSCut limitations are not dominated by simple low-frequency drift alone.

# 2. Introduction

Large-scale time-domain astronomy missions such as TESS provide unprecedented opportunities for studying stellar variability. TESS continuously monitors large regions of the sky and produces massive quantities of photometric time-series data suitable for variable star analysis.

At the same time, the AAVSO Variable Star Index (VSX) provides an extensive catalog of variable star classifications. Combining VSX labels with TESS photometric observations enables large-scale supervised machine learning studies of stellar variability.

However, constructing such a dataset presents several challenges:

1. VSX objects do not map uniquely to TESS TIC identifiers.
2. Many variable stars do not have pipeline-generated SPOC light curves.
3. TESS observational coverage is incomplete and provenance-dependent.
4. Extracted TESSCut light curves often contain stronger systematics and lower photometric quality.
5. Variable star families exhibit very different temporal characteristics.

# 3. Data Sources

## 3.1 AAVSO VSX Catalog

Variable star labels were obtained from the AAVSO Variable Star Index (VSX). Families included in this work include:

- CEPHEID
- CV
- DSCT_SXPHE
- ECLIPSING
- ELLIPSOIDAL_ROT
- LONG_PERIOD
- RRLYR
- XRAY
- YSO

## 3.2 TESS Light Curve Sources

Three TESS provenance sources were used:

### SPOC

SPOC products are pipeline-generated TESS light curves with strong systematic correction and optimized apertures. They generally provide the highest-quality photometry but have limited target coverage.

### QLP

QLP products provide broader coverage than SPOC but generally have lower photometric quality.

### TESSCut

TESSCut is a coordinate-based extraction service built on top of TESS Full Frame Images (FFIs). Instead of relying on pre-generated pipeline light curves, TESSCut dynamically extracts pixel cutouts centered on arbitrary sky coordinates and allows users to generate custom light curves through aperture photometry.

In this project, TESSCut serves as a critical fallback mechanism for stars lacking SPOC or QLP products. It dramatically increases coverage, but extracted light curves often exhibit higher noise, aperture contamination, background uncertainty, sector-to-sector inconsistencies, and stronger instrumental systematics.

# 4. Data Pipeline

## 4.1 Pipeline Overview

The pipeline follows this high-level structure:

```text
VSX Catalog
    ↓
Multi-candidate TIC Crossmatch
    ↓
SPOC / QLP Availability Query
    ↓
Best Source Selection
    ↓
TESSCut Fallback
    ↓
FITS Storage + QC
    ↓
Trend Detection / Detrending
    ↓
Feature Extraction
    ↓
Machine Learning Classification
```

**Figure 1.** High-level overview of the VSX–TESS processing pipeline.

## 4.2 VSX–TIC Crossmatching

A naive nearest-neighbor approach produced very poor recovery rates, approximately 12%, because the nearest TIC object often lacked TESS light curves or did not correspond to the most useful target.

To address this, the pipeline uses a multi-candidate crossmatching strategy:

1. Identify up to five nearby TIC candidates for each VSX object.
2. Query availability of SPOC and QLP products.
3. Select the best available source hierarchically:

$$
\text{SPOC} \rightarrow \text{QLP} \rightarrow \text{TESSCut fallback}
$$

## 4.3 Provenance Recovery Statistics

The final pipeline achieved approximately 96.5% usable light curve coverage.

Known recovery summary:

| Source | Count | Percent |
|---|---:|---:|
| TESSCut | 5034 | 68.30% |
| SPOC | 467 | 6.34% |
| QLP | 1612 | 21.87% |
| Missing | 257 | 3.49% |

This is one of the strongest engineering results of the project: most usable data came from TESSCut fallback, showing that standard pipeline products alone would have produced a much smaller and more biased dataset.

In [ ]:
# Figure 2: Provenance recovery chart
# This cell can use either the known summary above or a local metadata parquet.

known_counts = pd.DataFrame({
    "provenance": ["TESSCut", "SPOC", "QLP", "Missing"],
    "count": [5034, 467, 1612, 257],
})
known_counts["percent"] = known_counts["count"] / known_counts["count"].sum() * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(known_counts["provenance"], known_counts["count"])
ax.set_title("Recovered Light Curves by Provenance")
ax.set_ylabel("Count")
ax.set_xlabel("Provenance")
for i, row in known_counts.iterrows():
    ax.text(i, row["count"], f"{row['percent']:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()

fig.savefig(FIGURE_DIR / "figure_2_provenance_recovery.png", dpi=200)

# 5. Trend Detection and Conditional Detrending

## 5.1 Motivation

Early Random Forest experiments showed that TESSCut light curves performed substantially worse than SPOC and QLP. One hypothesis was that TESSCut light curves contain stronger long-term systematics or low-frequency trends.

This motivated a trend detection and conditional detrending experiment focused only on TESSCut light curves.

## 5.2 Initial Lomb–Scargle Low-Frequency Detector

The first approach used low-frequency Lomb–Scargle power to identify possible long-term trends. This was attractive because Lomb–Scargle is already central to the project.

However, this approach produced unrealistically high trend rates, often exceeding 80–90%. The issue was that stitched multi-sector TESS light curves contain large temporal gaps, and LS low-frequency power became sensitive to sector gaps and broad curvature rather than true removable drift.

This became an important methodological finding:

> Lomb–Scargle analysis is highly effective for periodicity analysis, but it is overly sensitive for generic trend detection on short, gapped TESS segments.

## 5.3 Segment-Wise Robust Drift Detector

The final trend detector uses a segment-wise robust drift metric.

Each light curve is split into contiguous observing segments based on large time gaps. For each segment, a linear trend is fitted:

$$
\text{flux} = a + bt
$$

The total fitted drift across the segment is:

$$
\text{drift} = |b \times \text{segmentDuration}|
$$

Robust scatter is estimated using the Median Absolute Deviation:

$$
\text{MAD} = \text{median}(|x_i - \text{median}(x)|)
$$

The final drift strength metric is:

$$
\text{segmentDriftStrength} =
\frac{|\text{slope} \times \text{segmentDuration}|}
{1.4826 \times \text{MAD}}
$$

A segment is classified as drifting when:

$$
\text{segmentDriftStrength} \ge 7.0
$$

A full light curve is classified as drifting only if:

$$
\text{fractionSegmentsWithDrift} \ge 0.5
$$

This majority-based criterion prevents one anomalous segment from flagging the entire light curve.

In [ ]:
# Figure 3: Drift summary by provenance/family, if trend parquet is available

if TREND_PARQUET.exists():
    trend_df = pd.read_parquet(TREND_PARQUET)
    print("Loaded trend parquet:", TREND_PARQUET)
    print("Columns:", list(trend_df.columns))

    if "provenance" in trend_df.columns and "robustDriftDetected" in trend_df.columns:
        drift_by_source = (
            trend_df.assign(robustDriftDetected=trend_df["robustDriftDetected"].fillna(False).astype(bool))
            .groupby("provenance")["robustDriftDetected"]
            .agg(["count", "sum"])
            .reset_index()
        )
        drift_by_source["percent"] = drift_by_source["sum"] / drift_by_source["count"] * 100
        display(drift_by_source)

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(drift_by_source["provenance"], drift_by_source["percent"])
        ax.set_title("Robust Drift Detection Rate by Provenance")
        ax.set_ylabel("Drift detected (%)")
        ax.set_xlabel("Provenance")
        plt.tight_layout()
        plt.show()
        fig.savefig(FIGURE_DIR / "figure_3_drift_by_provenance.png", dpi=200)
else:
    print("Trend parquet not found. Update TREND_PARQUET path to generate this figure.")

## 5.4 Conditional Detrending

Conditional detrending was applied only to TESSCut light curves with detected robust drift. SPOC and QLP rows were left unchanged because they already include stronger pipeline correction.

The updated metadata includes columns such as:

- `detrended`
- `detrendLightCurvePath`
- `detrendResolvedRawFitsPath`
- robust drift statistics
- detrending thresholds and method information

The detrending process was separately verified by checking FITS existence, column consistency, header metadata, and flux changes.

## 5.5 Detrending Result

The detrending process was technically successful, but it did not materially improve Random Forest classification performance for TESSCut light curves.

This is an important negative result:

> TESSCut limitations are not dominated by simple removable low-frequency drift alone.

Likely remaining causes include:

- photometric noise
- aperture contamination
- background estimation uncertainty
- sector-level inconsistency
- noisy period recovery
- reduced signal separability

# 6. Feature Extraction

A large-scale feature extraction pipeline converts each light curve into machine-learning-ready numerical features.

Feature families include:

## 6.1 Lomb–Scargle Features

- best frequency
- best period
- peak power
- false alarm probability
- top-N peak frequencies
- period ratios
- power ratios

## 6.2 Statistical Features

- amplitude
- mean flux
- median flux
- standard deviation
- MAD
- skewness
- kurtosis
- percentile ranges
- tail asymmetry

## 6.3 Metadata / Quality Features

- provenance
- robust drift indicators
- segment counts
- finite-point statistics
- quality flags

In [ ]:
# Figure 4: Feature availability summary, if feature parquet is available

if FEATURE_PARQUET.exists():
    feat_df = pd.read_parquet(FEATURE_PARQUET)
    print("Loaded feature parquet:", FEATURE_PARQUET)
    print("Shape:", feat_df.shape)
    display(feat_df.head())

    missing_rate = feat_df.isna().mean().sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(missing_rate.index[::-1], missing_rate.values[::-1])
    ax.set_title("Top Feature Missingness Rates")
    ax.set_xlabel("Missing fraction")
    plt.tight_layout()
    plt.show()
    fig.savefig(FIGURE_DIR / "figure_4_feature_missingness.png", dpi=200)
else:
    print("Feature parquet not found. Update FEATURE_PARQUET path to generate feature summary.")

# 7. Machine Learning Analysis

## 7.1 Models

Two supervised models were evaluated:

### Logistic Regression

Used as a linear baseline.

### Random Forest

Used as the main nonlinear classifier because it handles heterogeneous features well and provides feature importance estimates.

## 7.2 Key Finding

Random Forest consistently outperformed Logistic Regression, but performance varied strongly by provenance:

- SPOC performed best.
- QLP performed moderately.
- TESSCut remained substantially more difficult.
- Conditional detrending did not materially improve TESSCut performance.

This suggests that provenance is a major driver of downstream classification quality.

## 7.3 Confusion Matrix Analysis

Confusion matrices are important because they show which families are confused with one another.

Recommended confusion matrices for the final paper:

1. Random Forest on all provenance sources.
2. SPOC-only Random Forest.
3. QLP-only Random Forest.
4. TESSCut-only Random Forest.
5. Conditional-detrended TESSCut Random Forest.

The most important comparison is raw TESSCut vs conditional-detrended TESSCut.

In [ ]:
# Placeholder: confusion matrix generation
# Update paths/column names to match your saved ML prediction parquet files.

from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score

PREDICTION_PARQUET = ANALYSIS_OUTPUT_DIR / "test_predictions.parquet"

if PREDICTION_PARQUET.exists():
    pred_df = pd.read_parquet(PREDICTION_PARQUET)
    print("Loaded predictions:", PREDICTION_PARQUET)
    display(pred_df.head())

    # Update these column names if needed
    y_true_col = "true_label"
    y_pred_col = "predicted_label"

    if y_true_col in pred_df.columns and y_pred_col in pred_df.columns:
        y_true = pred_df[y_true_col]
        y_pred = pred_df[y_pred_col]

        print("Accuracy:", accuracy_score(y_true, y_pred))
        print("Balanced accuracy:", balanced_accuracy_score(y_true, y_pred))

        fig, ax = plt.subplots(figsize=(8, 8))
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, xticks_rotation=90, ax=ax)
        ax.set_title("Random Forest Confusion Matrix")
        plt.tight_layout()
        plt.show()
        fig.savefig(FIGURE_DIR / "figure_5_rf_confusion_matrix.png", dpi=200)
    else:
        print("Prediction columns not found. Update y_true_col/y_pred_col.")
else:
    print("Prediction parquet not found. Replace PREDICTION_PARQUET with your saved prediction output.")

## 7.4 Feature Importance

Feature importance analysis helps connect ML performance back to physical signals.

Expected important features include:

- period
- LS peak power
- amplitude
- power ratios
- distribution shape features
- provenance/quality metadata

This is valuable because it shows whether the model is using physically meaningful variability features rather than only artifacts.

In [ ]:
# Placeholder: feature importance plot
# Update IMPORTANCE_CSV or IMPORTANCE_PARQUET to match your saved output.

IMPORTANCE_CSV = ANALYSIS_OUTPUT_DIR / "rf_feature_importance.csv"

if IMPORTANCE_CSV.exists():
    imp_df = pd.read_csv(IMPORTANCE_CSV)
    display(imp_df.head())

    # Expected columns: feature, importance
    if {"feature", "importance"}.issubset(imp_df.columns):
        top = imp_df.sort_values("importance", ascending=False).head(20)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(top["feature"][::-1], top["importance"][::-1])
        ax.set_title("Top Random Forest Feature Importances")
        ax.set_xlabel("Importance")
        plt.tight_layout()
        plt.show()
        fig.savefig(FIGURE_DIR / "figure_6_rf_feature_importance.png", dpi=200)
    else:
        print("Expected columns feature/importance not found.")
else:
    print("Feature importance file not found. Replace IMPORTANCE_CSV with your saved output.")

# 8. Discussion

Several conclusions have emerged.

## 8.1 Data Recovery Is a Major Contribution

The pipeline improved usable coverage from approximately 12% under naive mapping to approximately 96.5% after multi-candidate matching and TESSCut fallback.

## 8.2 Coverage Does Not Equal Quality

TESSCut dramatically improved coverage but did not provide the same ML performance as SPOC or QLP. This shows the tradeoff:

$$
\text{coverage} \neq \text{quality}
$$

## 8.3 Simple Detrending Is Not Enough

Conditional detrending successfully removed strong segment-wise drift but did not materially improve Random Forest performance. This suggests that TESSCut performance limitations are not primarily caused by simple low-frequency trends.

## 8.4 Provenance-Aware Modeling Is Important

SPOC, QLP, and TESSCut are not interchangeable data sources. Future models should explicitly account for provenance.

# 9. Recommended Figures for Final Paper

The final paper should include approximately 6–8 high-quality figures:

| Figure | Purpose |
|---|---|
| Pipeline diagram | Explains full workflow |
| Provenance recovery chart | Shows data recovery contribution |
| Raw vs detrended TESSCut example | Shows detrending mechanics |
| Drift detection summary | Shows trend methodology |
| LS periodogram example | Shows periodicity feature extraction |
| Confusion matrix | Shows classification behavior |
| Feature importance plot | Shows model interpretation |
| Provenance performance comparison | Shows SPOC/QLP/TESSCut differences |

These figures should become core results, not merely decoration.

# 10. Future Work

Promising next directions include:

1. Provenance-aware modeling.
2. Family-specific preprocessing strategies.
3. Simplified SPOC-style cotrending for TESSCut.
4. Phase-folded features or deep learning.
5. Misclassification analysis using astrophysical context.
6. Careful comparison between VSX labels and model-predicted classes.

# 11. Conclusion

This project developed a scalable and provenance-aware pipeline for variable star classification using VSX labels and TESS light curves.

Major accomplishments include:

- multi-candidate VSX–TIC crossmatching
- hierarchical SPOC / QLP / TESSCut recovery
- large-scale FITS storage and QC
- trend detection and conditional detrending
- feature extraction using Lomb–Scargle and statistical descriptors
- Logistic Regression and Random Forest classification
- provenance-dependent model analysis

The key scientific finding is that TESSCut dramatically improves coverage but remains substantially weaker for classification, and simple conditional detrending does not fully close that gap. This suggests that the limiting factors are deeper photometric/systematic issues rather than simple removable low-frequency drift.